# Spot Browser

Browse `segvol.h5` frames to pick isolated diffraction-spot candidates.

- `segvol` stores threshold-segmented signal; pixel values are threshold levels, not grain IDs.
- Non-zero pixels = detected diffraction signal. Each connected component = one spot blob.
- Seg orientation is corrected with `rot90(frame[::-1], k=-1)`; raw frames need no correction.

## Workflow
1. **Setup** — set `SCAN`
2. **Browse** — set `FRAME`, run loading + figure cells
3. **Hover** blob map → read `blob_id`
4. **Zoom** — inspect seg + raw crop
5. **Confirm** — save to `confirmed_spots.h5`
6. **List** — print what has been saved so far

In [7]:
from pathlib import Path
from collections import deque
import numpy as np
import h5py
import hdf5plugin
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SEG_KEY = "segvol"

## 1 - Setup

In [2]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
ROOT    = Path("../../data_esrf/")
SCAN    = "Al"
RAW_KEY = "instrument/detector_0/data"
# ─────────────────────────────────────────────────────────────────────────────

SCAN_DIR = ROOT / SCAN
SEG_FILE = SCAN_DIR / "segvol.h5"
RAW_FILE = next(
    (p for p in sorted(SCAN_DIR.glob("*.h5"))
     if p.name != "segvol.h5" and "dark" not in p.name.lower()),
    None
)

print(f"Scan dir : {SCAN_DIR}")
print(f"Seg file : {SEG_FILE}  exists={SEG_FILE.exists()}")
print(f"Raw file : {RAW_FILE}  exists={RAW_FILE.exists() if RAW_FILE else False}")

with h5py.File(SEG_FILE, "r") as f:
    N_FRAMES, H, W = f[SEG_KEY].shape
print(f"Frames: {N_FRAMES}  ({H}x{W})")

Scan dir : ../../data_esrf/Al
Seg file : ../../data_esrf/Al/segvol.h5  exists=True
Raw file : ../../data_esrf/Al/_iris_iris_500_cont.h5  exists=True
Frames: 3600  (2048x2048)


## 2 - Helpers

In [3]:
def fix_orientation(frame):
    return np.rot90(frame[::-1], k=-1)


def _norm_frame(frame_idx):
    return int(frame_idx) % N_FRAMES


SEG_CACHE = {}
RAW_FRAME_CACHE = {}
BLOB_CACHE = {}


def _remember(cache, key, value, maxsize):
    cache[key] = value
    while len(cache) > maxsize:
        cache.pop(next(iter(cache)))
    return value


def load_seg(frame_idx):
    frame_idx = _norm_frame(frame_idx)
    if frame_idx in SEG_CACHE:
        return SEG_CACHE[frame_idx]
    with h5py.File(SEG_FILE, "r") as f:
        frame = fix_orientation(np.asarray(f[SEG_KEY][frame_idx]))
    return _remember(SEG_CACHE, frame_idx, frame, 64)


def load_seg_many(frame_indices):
    """Load several segvol frames with one HDF5 open/cache lifetime.

    Contiguous requests are read as slices so gzip chunks are decompressed once
    per range instead of once per individual frame.
    """
    unique = sorted({_norm_frame(frame_idx) for frame_idx in frame_indices})
    frames = {}
    missing = [idx for idx in unique if idx not in SEG_CACHE]

    if missing:
        ranges = []
        start = prev = missing[0]
        for idx in missing[1:]:
            if idx == prev + 1:
                prev = idx
            else:
                ranges.append((start, prev + 1))
                start = prev = idx
        ranges.append((start, prev + 1))

        with h5py.File(SEG_FILE, "r") as f:
            data = f[SEG_KEY]
            for start, stop in ranges:
                batch = np.asarray(data[start:stop])
                for offset, frame in enumerate(batch):
                    idx = start + offset
                    _remember(SEG_CACHE, idx, fix_orientation(frame), 64)

    for idx in unique:
        frames[idx] = SEG_CACHE[idx]
    return frames


def load_raw(frame_idx):
    if RAW_FILE is None:
        return None
    frame_idx = _norm_frame(frame_idx)
    if frame_idx in RAW_FRAME_CACHE:
        return RAW_FRAME_CACHE[frame_idx]
    with h5py.File(RAW_FILE, "r") as f:
        frame = np.asarray(f[RAW_KEY][frame_idx])
    return _remember(RAW_FRAME_CACHE, frame_idx, frame, 16)


try:
    from scipy import ndimage as _ndi
except ImportError:
    _ndi = None


def compute_blobs(seg_frame):
    """Connected components on the binary (non-zero) mask.
    Returns blob_map (int32, 0=background, 1..N=blob IDs) and n_blobs.

    If scipy is installed this uses scipy.ndimage.label, which is much faster
    for 2048x2048 frames. Otherwise it falls back to the original pure-Python
    flood fill.
    """
    binary = seg_frame > 0
    if _ndi is not None:
        structure = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=bool)
        blob_map, n_blobs = _ndi.label(binary, structure=structure)
        return blob_map.astype(np.int32, copy=False), int(n_blobs)

    H, W = binary.shape
    blob_map = np.zeros((H, W), dtype=np.int32)
    label_id = 0
    for start_r, start_c in zip(*np.where(binary)):
        start_r, start_c = int(start_r), int(start_c)
        if blob_map[start_r, start_c] > 0:
            continue
        label_id += 1
        q = deque([(start_r, start_c)])
        blob_map[start_r, start_c] = label_id
        while q:
            r, c = q.popleft()
            for dr, dc in ((-1,0),(1,0),(0,-1),(0,1)):
                nr, nc = r+dr, c+dc
                if 0 <= nr < H and 0 <= nc < W and binary[nr,nc] and blob_map[nr,nc] == 0:
                    blob_map[nr,nc] = label_id
                    q.append((nr,nc))
    return blob_map, label_id


def blob_state(frame_idx):
    frame_idx = _norm_frame(frame_idx)
    if frame_idx in BLOB_CACHE:
        return BLOB_CACHE[frame_idx]
    seg_frame = load_seg(frame_idx)
    blob_map, n_blobs = compute_blobs(seg_frame)
    return _remember(BLOB_CACHE, frame_idx, (seg_frame, blob_map, n_blobs), 32)


def remember_blob_state(frame_idx, seg_frame, blob_map, n_blobs):
    frame_idx = _norm_frame(frame_idx)
    _remember(SEG_CACHE, frame_idx, seg_frame, 64)
    return _remember(BLOB_CACHE, frame_idx, (seg_frame, blob_map, int(n_blobs)), 32)


def frame_state(frame_idx, include_raw=True):
    frame_idx = _norm_frame(frame_idx)
    seg_frame, blob_map, n_blobs = blob_state(frame_idx)
    raw_frame = load_raw(frame_idx) if include_raw else None
    return {
        "frame": frame_idx,
        "seg": seg_frame,
        "blob_map": blob_map,
        "n_blobs": n_blobs,
        "raw": raw_frame,
    }


print("Helpers defined.")
print("Tip: installing scipy enables a much faster connected-component backend.")


Helpers defined.
Tip: installing scipy enables a much faster connected-component backend.


## 3 - Browse a frame

Set `FRAME` and re-run the cells below.

**Figure 1** — blob map (left, hover -> blob_id) and raw threshold values (right).  
**Figure 2** — 3x2 Friedel neighbourhood. A clean spot appears compact in 1-3 frames and has a matching blob in the Friedel column.

In [ ]:
# this shows multiple neighbouring frames
# Stack as (6, H, W): rows = prev/main/next, cols = main/Friedel
frames_stack = np.stack([
    seg_prev,  seg_fprev,
    seg_main,  seg_fmain,
    seg_next,  seg_fnext,
])

fig2 = px.imshow(
    frames_stack,
    facet_col=0,
    facet_col_wrap=2,
    origin="upper",
    color_continuous_scale="Turbo",
)

# Rename facet labels
labels = {
    "0": f"prev  (frame {FRAME-1})",
    "1": f"Friedel prev  (frame {FRIEDEL-1})",
    "2": f"main  (frame {FRAME})",
    "3": f"Friedel main  (frame {FRIEDEL})",
    "4": f"next  (frame {FRAME+1})",
    "5": f"Friedel next  (frame {FRIEDEL+1})",
}
for ann in fig2.layout.annotations:
    ann.text = labels.get(ann.text, ann.text)

fig2.update_layout(
    title=f"{SCAN}  -  Friedel neighbourhood  (offset={FRIEDEL_OFFSET})",
    width=1100, height=1650, margin=dict(l=60, r=20, t=80, b=40),
)
fig2.show(renderer='browser')

NameError: name 'seg_prev' is not defined

In [4]:
# ── CONTROLS ──────────────────────────────────────────────────────────────────
FRAME          = 208
FRIEDEL_OFFSET = 1800
# ─────────────────────────────────────────────────────────────────────────────

FRAME = _norm_frame(FRAME)
FRIEDEL = _norm_frame(FRAME + FRIEDEL_OFFSET)
print(f"Loading ...  main={FRAME}  Friedel={FRIEDEL}")

needed = [FRAME - 1, FRAME, FRAME + 1, FRIEDEL - 1, FRIEDEL, FRIEDEL + 1]
loaded = load_seg_many(needed)

seg_prev  = loaded[_norm_frame(FRAME - 1)]
seg_main  = loaded[FRAME]
seg_next  = loaded[_norm_frame(FRAME + 1)]
seg_fprev = loaded[_norm_frame(FRIEDEL - 1)]
seg_fmain = loaded[FRIEDEL]
seg_fnext = loaded[_norm_frame(FRIEDEL + 1)]
raw_main  = load_raw(FRAME)

print("Computing blobs ...")
blob_map, n_blobs = compute_blobs(seg_main)
remember_blob_state(FRAME, seg_main, blob_map, n_blobs)
print(f"Done.  {n_blobs} blobs in frame {FRAME}")


Loading ...  main=208  Friedel=2008
Computing blobs ...
Done.  130 blobs in frame 208


In [5]:
# ── Figure 1: blob map (hover) + thresholded seg (fast PNG) ──────────────────
fig1 = make_subplots(
    rows=1, cols=2,
    column_titles=[
        f"Blobs - frame {FRAME}  (hover -> blob_id)",
        f"Seg threshold - frame {FRAME}",
    ],
    horizontal_spacing=0.04,
)

# Blob map: Heatmap so hover shows blob_id
fig1.add_trace(go.Heatmap(
    z=blob_map, colorscale="Turbo", showscale=False, name="blobs",
    hovertemplate="col=%{x}  row=%{y}<br><b>blob_id=%{z}</b><extra></extra>",
    zmin=0,
), row=1, col=1)

# Seg threshold: binary PNG — much faster for 2048×2048
fig1.add_trace(
    px.imshow(seg_main, origin="upper",
              color_continuous_scale="Turbo").data[0],
    row=1, col=2,
)

fig1.update_yaxes(autorange="reversed")
fig1.update_layout(xaxis2=dict(matches="x"), yaxis2=dict(matches="y"))
fig1.update_layout(
    title=f"{SCAN}  -  frame {FRAME}  ({n_blobs} blobs)",
    width=1100, height=550,
    margin=dict(l=40, r=20, t=60, b=40),
)
fig1.show(renderer='browser')

## 4 - Zoom into a blob

Hover the **Blobs panel** in Figure 1 to read `blob_id`, enter it below.

In [6]:
# ── CONTROLS ──────────────────────────────────────────────────────────────────
BLOB_ID = 44
PAD     = 10
# ─────────────────────────────────────────────────────────────────────────────

state = frame_state(FRAME, include_raw=True)
FRAME = state["frame"]
blob_map = state["blob_map"]
raw_main = state["raw"]

rows, cols = np.where(blob_map == BLOB_ID)
if len(rows) == 0:
    print(f"Blob {BLOB_ID} not found in frame {FRAME}.")
else:
    r0 = max(0, rows.min()-PAD);  r1 = min(H, rows.max()+PAD+1)
    c0 = max(0, cols.min()-PAD);  c1 = min(W, cols.max()+PAD+1)
    print(f"Frame {FRAME}  blob {BLOB_ID}")
    print(f"Blob pixels rows {rows.min()}-{rows.max()}  cols {cols.min()}-{cols.max()}  area={len(rows)} px")
    print(f"Crop bbox rows {r0}-{r1}, cols {c0}-{c1}")

    seg_crop = blob_map[r0:r1, c0:c1]
    raw_crop = raw_main[r0:r1, c0:c1] if raw_main is not None else None

    has_raw   = raw_crop is not None
    n_cols    = 2 if has_raw else 1
    subtitles = [f"Blob {BLOB_ID} - seg"] + ([f"Blob {BLOB_ID} - raw"] if has_raw else [])

    fig3 = make_subplots(rows=1, cols=n_cols, subplot_titles=subtitles,
                         horizontal_spacing=0.06)

    fig3.add_trace(
        px.imshow(seg_crop, origin="upper",
                  color_continuous_scale="Turbo").data[0],
        row=1, col=1,
    )
    if has_raw:
        fig3.add_trace(
            px.imshow(raw_crop, origin="upper",
                      color_continuous_scale="gray").data[0],
            row=1, col=2,
        )

    fig3.update_layout(
        title=f"{SCAN}  frame {FRAME}  blob {BLOB_ID}  {rows.max()-rows.min()+1}x{cols.max()-cols.min()+1} px  area={len(rows)}",
        height=550, width=1100, margin=dict(l=40, r=20, t=60, b=40),
    )
    fig3.show(renderer='browser')


Frame 208  blob 44
Blob pixels rows 847-918  cols 251-338  area=3545 px
Crop bbox rows 837-929, cols 241-349


## 5 - Confirm spot

If the zoom looks clean and isolated, set `CONFIRM = True` and run.

In [48]:
# ── CONTROLS ──────────────────────────────────────────────────────────────────
CONFIRM   = True
OVERWRITE = True  # set True to replace an existing saved spot with corrected bbox/mask
PAD_SAVE  = 10
# ─────────────────────────────────────────────────────────────────────────────

OUT_FILE = ROOT.parent / "../data_esrf/confirmed_spots_merged.h5"

state = frame_state(FRAME, include_raw=False)
FRAME = state["frame"]
blob_map = state["blob_map"]

rows, cols = np.where(blob_map == BLOB_ID)
if len(rows) == 0:
    print(f"Blob {BLOB_ID} not found in frame {FRAME} - check the frame/blob_id from the hover text.")
else:
    r0 = max(0, rows.min()-PAD_SAVE);  r1 = min(H, rows.max()+PAD_SAVE+1)
    c0 = max(0, cols.min()-PAD_SAVE);  c1 = min(W, cols.max()+PAD_SAVE+1)
    mask = (blob_map[r0:r1, c0:c1] == BLOB_ID)
    bbox = np.array([r0, c0, r1, c1], dtype=np.int32)
    group = f"{SCAN}/frame_{FRAME:04d}_blob_{BLOB_ID:04d}"

    print(f"Spot  : {group}")
    print(f"Bbox  : rows {r0}-{r1},  cols {c0}-{c1}")
    print(f"Mask  : {mask.shape},  True pixels = {mask.sum()}")

    if CONFIRM:
        with h5py.File(OUT_FILE, "a") as f:
            if group in f:
                old_bbox = f[group]["bbox"][()].tolist() if "bbox" in f[group] else None
                if not OVERWRITE:
                    print(f"Already exists with bbox={old_bbox}.")
                    print("Set OVERWRITE = True to replace it with the bbox/mask printed above.")
                else:
                    del f[group]
                    print(f"Overwriting existing spot. Old bbox={old_bbox}")

            if group not in f:
                g = f.create_group(group)
                g.create_dataset("mask", data=mask, compression="gzip")
                g.create_dataset("bbox", data=bbox)
                g.attrs["scan"]      = SCAN
                g.attrs["frame"]     = FRAME
                g.attrs["blob_id"]   = BLOB_ID
                g.attrs["pad"]       = PAD_SAVE
                g.attrs["overwrite"] = bool(OVERWRITE)
                print(f"Saved -> {OUT_FILE}  [{group}]")
    else:
        print("(dry run - set CONFIRM = True to save)")


Spot  : Ti7Al/frame_1605_blob_0028
Bbox  : rows 852-915,  cols 363-403
Mask  : (63, 40),  True pixels = 421
Saved -> ../../data_esrf/../data_esrf/confirmed_spots_merged.h5  [Ti7Al/frame_1605_blob_0028]


## 6 - List confirmed spots

In [49]:
OUT_FILE = ROOT.parent / "../data_esrf/confirmed_spots_merged.h5"

if not OUT_FILE.exists():
    print("No confirmed_spots.h5 yet.")
else:
    with h5py.File(OUT_FILE, "r") as f:
        entries = []
        def _collect(name, obj):
            if isinstance(obj, h5py.Group) and "mask" in obj:
                entries.append({
                    "scan":    obj.attrs.get("scan", "?"),
                    "frame":   obj.attrs.get("frame", "?"),
                    "blob_id": obj.attrs.get("blob_id", "?"),
                    "area_px": int(obj["mask"][()].sum()),
                    "bbox":    obj["bbox"][()].tolist(),
                })
        f.visititems(_collect)

    by_scan = {}
    for entry in entries:
        scan = str(entry["scan"])
        by_scan[scan] = by_scan.get(scan, 0) + 1

    print(f"{len(entries)} confirmed spot(s) in {OUT_FILE.name}\n")
    print("By scan:")
    for scan, count in sorted(by_scan.items()):
        print(f"  {scan}: {count}")
    print()

    print(f"{'scan':<20} {'frame':>6} {'blob_id':>8} {'area_px':>8}  bbox [r0,c0,r1,c1]")
    print("-" * 72)
    for e in entries:
        print(f"{e['scan']:<20} {e['frame']:>6} {e['blob_id']:>8} {e['area_px']:>8}  {e['bbox']}")


172 confirmed spot(s) in confirmed_spots_merged.h5

By scan:
  Al: 35
  Al_big_grains: 28
  Al_small_grains: 13
  Cu: 10
  IN718_twins: 32
  Iron: 32
  Iron_deformed: 5
  Ti7Al: 17

scan                  frame  blob_id  area_px  bbox [r0,c0,r1,c1]
------------------------------------------------------------------------
Al                     1100       84    10451  [1648, 119, 1791, 286]
Al                     1101        1     3088  [77, 1566, 158, 1654]
Al                     1101        6     1981  [517, 1112, 590, 1189]
Al                     1102       85    10845  [1130, 1775, 1270, 1950]
Al                     1103       10     4252  [358, 1737, 457, 1842]
Al                     1104        2     3974  [224, 1581, 310, 1683]
Al                     1104       22     3129  [816, 1429, 881, 1559]
Al                     1104       49     1750  [1055, 1489, 1122, 1562]
Al                     1106       37     1756  [342, 1026, 409, 1097]
Al                     1106       38      731 